In [1]:
import xarray as xr
from obstore.store import from_url

from virtualizarr import open_virtual_mfdataset
from virtualizarr.parsers import HDFParser
from virtualizarr.registry import ObjectStoreRegistry
from distributed import Client

In [2]:
client = Client(n_workers=16)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: https://cluster-shuuf.dask.host/jupyter/proxy/8787/status,
Dashboard: https://cluster-shuuf.dask.host/jupyter/proxy/8787/status,Workers: 32
Total threads: 32,Total memory: 60.69 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:32771,Workers: 0
Dashboard: https://cluster-shuuf.dask.host/jupyter/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:34521,Total threads: 1
Dashboard: https://cluster-shuuf.dask.host/jupyter/proxy/41749/status,Memory: 1.90 GiB
Nanny: tcp://127.0.0.1:41405,


In [3]:
drop_vars = [
    "gw",
    "hyam",
    "hybm",
    "P0",
    "hyai",
    "hybi",
    "ndbase",
    "nsbase",
    "nbdate",
    "nbsec",
    "mdt",
    "date",
    "datesec",
    "time_bnds",
    "date_written",
    "time_written",
    "ndcur",
    "nscur",
    "co2vmr",
    "ch4vmr",
    "n2ovmr",
    "f11vmr",
    "f12vmr",
    "sol_tsi",
    "nsteph",
]

In [4]:
bucket = "s3://ncar-cesm2-arise/"
store = from_url(bucket, region="us-east-2", skip_signature=True)
registry = ObjectStoreRegistry({bucket: store})

parser = HDFParser(drop_variables=drop_vars)

In [10]:
# works for 06
year_range_dict = {
    "2035": "20350101-20441231",  # 10yrs
    "2045": "20450101-20541231",  # 10yrs
    "2055": "20550101-20641231",  # 10yrs
    "2065": "20650101-20691231",  # 5yrs
}


simulation_id = "006"
var_list = ["T", "TS", "PRECC", "PRECT", "U", "V"]


def create_paths(*, var_list: list[str], year_range_dict: dict, simulation_id: str):
    return [
        f"s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.{simulation_id}/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.{simulation_id}.cam.h1.{var}.{year_str}.nc"
        for var in var_list
        for _, year_str in year_range_dict.items()
    ]


urls = create_paths(
    var_list=var_list, year_range_dict=year_range_dict, simulation_id=simulation_id
)

In [11]:
urls

['s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.T.20350101-20441231.nc',
 's3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.T.20450101-20541231.nc',
 's3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.T.20550101-20641231.nc',
 's3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.T.20650101-20691231.nc',
 's3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.TS.20350101-20441231.nc',
 's3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17

In [12]:
# # simulation_ids = ['006','007','008','009','010']
# urls = [
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.T.20350101-20441231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.T.20450101-20541231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.T.20550101-20641231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.T.20650101-20691231.nc",

# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.TS.20350101-20441231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.TS.20450101-20541231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.TS.20550101-20641231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.TS.20650101-20691231.nc",

# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.PRECC.20350101-20441231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.PRECC.20450101-20541231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.PRECC.20550101-20641231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.PRECC.20650101-20691231.nc",

# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.PRECT.20350101-20441231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.PRECT.20450101-20541231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.PRECT.20550101-20641231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.PRECT.20650101-20691231.nc",

# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.U.20350101-20441231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.U.20450101-20541231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.U.20550101-20641231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.U.20650101-20691231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.V.20350101-20441231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.V.20450101-20541231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.V.20550101-20641231.nc",
# "s3://ncar-cesm2-arise/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006/atm/proc/tseries/day_1/b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006.cam.h1.V.20650101-20691231.nc",
# ]

In [13]:
combined_vds = open_virtual_mfdataset(
    urls,
    registry=registry,
    parser=parser,
    combine="by_coords",
    combine_attrs="drop_conflicts",
    loadable_variables=["time", "lev", "lat", "lon", "ilev"],
    parallel="dask",
)
combined_vds
# NotImplementedError: Unsupported indexer. So-called 'fancy indexing' via numpy arrays is not supported, but received [ 0  1  2 ... -1 -1 -1]

/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:164: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:164: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:164: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:164: UserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/opt/coiled/env/lib/python3.13/site-packages/numcodecs/zarr3.py:

<xarray.Dataset> Size: 602GB
Dimensions:  (time: 12776, lev: 70, lat: 192, lon: 288, ilev: 71)
Coordinates:
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB 0.0 1.25 2.5 3.75 5.0 ... 355.0 356.2 357.5 358.8
  * lev      (lev) float64 560B 5.96e-06 9.827e-06 1.62e-05 ... 976.3 992.6
  * ilev     (ilev) float64 568B 4.5e-06 7.42e-06 1.223e-05 ... 985.1 1e+03
  * time     (time) object 102kB 2035-01-01 00:00:00 ... 2070-01-01 00:00:00
Data variables:
    T        (time, lev, lat, lon) float32 198GB ManifestArray<shape=(12776, ...
    TS       (time, lat, lon) float32 3GB ManifestArray<shape=(12776, 192, 28...
    PRECC    (time, lat, lon) float32 3GB ManifestArray<shape=(12776, 192, 28...
    PRECT    (time, lat, lon) float32 3GB ManifestArray<shape=(12776, 192, 28...
    U        (time, lev, lat, lon) float32 198GB ManifestArray<shape=(12776, ...
    V        (time, lev, lat, lon) float32 198GB ManifestArray<shape=(12776, ...
Attributes:
    Conventions:       CF-1.0
    source:            CAM
    case:              b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.006
    logname:           geostrat
    initial_file:      b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.001.c...
    topography_file:   /glade/p/cesmdata/cseg/inputdata/atm/cam/topo/fv_0.9x1...
    model_doi_url:     https://doi.org/10.5065/D67H1H0V
    time_period_freq:  day_1

In [14]:
combined_vds.ilev

<xarray.DataArray 'ilev' (ilev: 71)> Size: 568B
array([4.500500e-06, 7.420100e-06, 1.223370e-05, 2.017000e-05, 3.325450e-05,
       5.482750e-05, 9.039800e-05, 1.490400e-04, 2.457200e-04, 4.051250e-04,
       6.679400e-04, 1.101265e-03, 1.815650e-03, 2.993500e-03, 4.963000e-03,
       8.150651e-03, 1.347700e-02, 2.231900e-02, 3.679650e-02, 6.066500e-02,
       9.915650e-02, 1.573900e-01, 2.388500e-01, 3.452000e-01, 4.751350e-01,
       6.318050e-01, 8.291550e-01, 1.082740e+00, 1.406850e+00, 1.818850e+00,
       2.339800e+00, 2.995050e+00, 3.814700e+00, 4.834450e+00, 6.096350e+00,
       7.649350e+00, 9.550100e+00, 1.186400e+01, 1.466550e+01, 1.803800e+01,
       2.207550e+01, 2.688250e+01, 3.257350e+01, 3.927300e+01, 4.711450e+01,
       5.624050e+01, 6.680050e+01, 8.070142e+01, 9.494104e+01, 1.116932e+02,
       1.314013e+02, 1.545868e+02, 1.818634e+02, 2.139528e+02, 2.517044e+02,
       2.961172e+02, 3.483666e+02, 4.098352e+02, 4.821499e+02, 5.672244e+02,
       6.523330e+02, 7.304459e+02, 7.963631e+02, 8.453537e+02, 8.737159e+02,
       9.003246e+02, 9.249645e+02, 9.474323e+02, 9.675386e+02, 9.851122e+02,
       1.000000e+03])
Coordinates:
  * ilev     (ilev) float64 568B 4.5e-06 7.42e-06 1.223e-05 ... 985.1 1e+03
Attributes:
    long_name:      hybrid level at interfaces (1000*(A+B))
    units:          hPa
    positive:       down
    standard_name:  atmosphere_hybrid_sigma_pressure_coordinate
    formula_terms:  a: hyai b: hybi p0: P0 ps: PS

In [15]:
# we have atmosphere_hybrid_sigma_pressure_coordinate - convert?
# ilev vs lev

<xarray.DataArray 'lev' (lev: 70)> Size: 560B
array([5.960300e-06, 9.826900e-06, 1.620185e-05, 2.671225e-05, 4.404100e-05,
       7.261275e-05, 1.197190e-04, 1.973800e-04, 3.254225e-04, 5.365325e-04,
       8.846025e-04, 1.458457e-03, 2.404575e-03, 3.978250e-03, 6.556826e-03,
       1.081383e-02, 1.789800e-02, 2.955775e-02, 4.873075e-02, 7.991075e-02,
       1.282732e-01, 1.981200e-01, 2.920250e-01, 4.101675e-01, 5.534700e-01,
       7.304800e-01, 9.559475e-01, 1.244795e+00, 1.612850e+00, 2.079325e+00,
       2.667425e+00, 3.404875e+00, 4.324575e+00, 5.465400e+00, 6.872850e+00,
       8.599725e+00, 1.070705e+01, 1.326475e+01, 1.635175e+01, 2.005675e+01,
       2.447900e+01, 2.972800e+01, 3.592325e+01, 4.319375e+01, 5.167750e+01,
       6.152050e+01, 7.375096e+01, 8.782123e+01, 1.033171e+02, 1.215472e+02,
       1.429940e+02, 1.682251e+02, 1.979081e+02, 2.328286e+02, 2.739108e+02,
       3.222419e+02, 3.791009e+02, 4.459926e+02, 5.246872e+02, 6.097787e+02,
       6.913894e+02, 7.634045e+02, 8.208584e+02, 8.595348e+02, 8.870202e+02,
       9.126445e+02, 9.361984e+02, 9.574855e+02, 9.763254e+02, 9.925561e+02])
Coordinates:
  * lev      (lev) float64 560B 5.96e-06 9.827e-06 1.62e-05 ... 976.3 992.6
Attributes:
    long_name:      hybrid level at midpoints (1000*(A+B))
    units:          hPa
    positive:       down
    standard_name:  atmosphere_hybrid_sigma_pressure_coordinate
    formula_terms:  a: hyam b: hybm p0: P0 ps: PS

In [ ]:
combined_vds_with_sim = combined_vds.expand_dims(simulation_id=[simulation_id])

In [ ]:
combined_vds_with_sim

In [ ]:
import icechunk


config = icechunk.RepositoryConfig.default()
config.set_virtual_chunk_container(
    icechunk.VirtualChunkContainer(
        url_prefix=bucket,
        store=icechunk.s3_store(region="us-east-2", anonymous=True),
    ),
)

# create an in-memory icechunk repository that includes the virtual chunk containers
storage = icechunk.in_memory_storage()
repo = icechunk.Repository.create(storage, config)

# open a writable icechunk session to be able to add new contents to the store
session = repo.writable_session("main")

# write the virtual dataset to the session's IcechunkStore instance, using VirtualiZarr's `.vz` accessor
combined_vds.vz.to_icechunk(session.store)

# commit your changes so that they are permanently available as a new snapshot
snapshot_id = session.commit("Wrote first dataset")
print(snapshot_id)

# optionally persist the new permissions to be permanent, which you probably want
# otherwise every user who wants to read the referenced virtual data back later will have to repeat the `config.set_virtual_chunk_container` step at read time.
repo.save_config()

In [ ]:
session = repo.readonly_session("main")

rtds = xr.open_zarr(session.store, consolidated=False)

In [ ]:
rtds.isel(lev=0).nbytes / 1e9
# surface level , 6 vars